In [1]:
import os
import json
from dotenv import load_dotenv
load_dotenv()
from fewshot import examples

from langchain_upstage import ChatUpstage, UpstageEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder , FewShotPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

In [2]:
def get_llm(model = "solar-1-mini-chat"):
    return ChatUpstage(model = model)

In [3]:
def get_rag_chain():
    example_prompt = ChatPromptTemplate.from_messages(
        [
            ("human","{input}"),
            ("ai", "{output}")
        ]
    )

    few_shot_prompt = FewShotPromptTemplate(
        examples=examples,
        example_prompt=example_prompt
    )


    analysis_s_prompt = """
당신은 학생들에 실력을 분석하는 챗봇입니다.
학생들의 실력을 분석할때는 아래에 학생데이터 student_data를 활용하여 답변해주세요.
출력은 학생에 실력 총 점수와 , 그렇게 점수를 매긴 이유를 출력하세요.

student_data : {student_data}
"""



    s_prompt = ChatPromptTemplate.from_messages(
        [
            ("system",analysis_s_prompt),
            few_shot_prompt,
            ("human","{input}")
        ]
    )


    return s_prompt

In [15]:
def get_test():
    s_prompt = get_rag_chain()
    response = s_prompt.invoke({"student","""  {{
    "name": "김민준",
    "skills": {{
      "python": 4,
      "ml": 3,
      "java": 2
    }},
    "goal": "AI",
    "experience": [
      "FastAPI 기반 챗봇 서버 구현 및 KoBERT 모델 연동",
      "사용자 질문 로그 수집 후 간단한 intent 분류 모델 개선",
      "프로젝트 배포를 위해 Docker로 컨테이너 구성"
    ],
    "collaboration": 3
  }},
"""})
    print(response.content)


In [16]:
get_test()

KeyError: 'template'